# ActivitySCOPE — Does $\Delta H$ replace $\Delta Q$ and $S_{\rm EV}$?

This notebook is self-contained: it loads the databases, trains the three models, and
then runs six experiments designed to answer four specific questions.

| Question | Answered by |
|---|---|
| **Q1.** Does $\Delta H$ separate active from background as well as $\Delta Q$? | Run 2 (decisive), Run 3 (supporting) |
| **Q2.** Is there a more principled definition than the 0.5 tolerance? | Run 0 (validity), Run 1 (behaviour) |
| **Q3.** How much simpler would the paper be? | Run 6 |
| **Q4.** Can it find objects we previously missed? | Run 4, Run 5 |

## The definition being tested

The current cell 22 in *Start Here* reports the $\Delta H$ at which $E[N_{\rm opp}]$ falls to
$N_{\rm obs} + 0.5$. That $0.5$ is in **oppositions, not magnitudes**, and because $\lambda(H)$
is monotone the solver always lands on the near edge of the tolerance band — so the
tolerance *is* the operating point, and it is scale-inconsistent ($+0.5$ is a 50% correction
at $N=1$ and 7% at $N=7$).

This notebook instead uses three crossing points that already exist in the method, none of
which introduces a new constant:

$$\Delta H_E = \min\{\delta \ge 0 : E[N_{\rm opp}](H+\delta) \le N_{\rm obs}\}$$
$$\Delta H_Q = \min\{\delta \ge 0 : Q_{0.006}(H+\delta) \le N_{\rm obs}\} \quad (\text{the point where } \Delta Q = 0)$$
$$\Delta H_P = \min\{\delta \ge 0 : P(N_{\rm opp}\ge4)(H+\delta) \le 0.5\}$$

with $\Delta H_Q \le \Delta H_E$ always, and a significance $Z_H = \Delta H / \sigma_H$ where
$\sigma_H \approx 0.45$ mag combines lightcurve half-amplitude, $G=0.15$ phase-function error,
and detection-limit bias in quadrature.

To first order $\Delta H_Q \approx \Delta Q\,/\,|dQ_{0.006}/dH|$ — i.e. $\Delta H$ is the
opposition deficit divided by its own sensitivity to the dominant systematic.

## Runtime

Set `QUICK = True` for a ~15 minute smoke test that will produce directionally useful
but noisy numbers. Set `QUICK = False` (~1.5–2 h) before believing any result.

In [1]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
import json
from scipy.stats import poisson, spearmanr
import matplotlib.pyplot as plt
import datetime
import os
import tempfile
import importlib

import activityscope_utils as utils
import h_neutrality as hn
import h_neutrality_eval as hev
importlib.reload(hn); importlib.reload(hev)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

/Users/petervw/mambaforge/envs/activityscope/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================ PARAMETERS ============================
QUICK = True          # <-- set False for the real run

TRAINING_TIME_LIMIT = 600 if QUICK else 4000

NUM_OPPS_FOR_TRAINING = 4
MIN_ARC_LENGTH_FOR_TRAINING = 20
MIN_NUM_OBS_FOR_TRAINING = 16

# --- Delta-H parameters -------------------------------------------------
SIGMA_H     = 0.45    # adopted 1-sigma on a catalog H_V (see markdown above)
DELTA_MAX   = 15.0    # brightening beyond this is reported as right-censored
N_BISECT    = 12      # 15 mag / 2^12 -> 0.004 mag; report only to 0.1 mag
QUANTILE_LEVEL = 0.006
P_STAR      = 0.5

# Pool sizes (raise for the real run)
N_CANDIDATES_RUN1  = 150 if QUICK else 250
N_NEGATIVES_RUN2   = 1500 if QUICK else 4000
N_WIDE_RUN4        = 1500 if QUICK else 4000
RNG_SEED = 0

RESULTS_DIR = "deltaH_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open('known_active_objects.json', 'r') as f:
    known_strongly_suspected_active_objects = json.load(f)
with open('dual_designation_list.json', 'r') as f:
    dual_designation_list = json.load(f)

print(f"QUICK={QUICK}  training time limit={TRAINING_TIME_LIMIT}s")

QUICK=True  training time limit=600s


## Part 1 — Load databases and engineer features

Identical to *Start Here* cells 3–6.

In [3]:
try:
    orb = utils.load_all_databases()
except Exception as e:
    print(f"Error loading databases: {e}")
    cache_dir = os.path.join(os.getcwd(), '.cache')
    for filename in os.listdir(cache_dir):
        file_path = os.path.join(cache_dir, filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e2:
            print(f"Error deleting file {file_path}: {e2}")
    raise Exception(f"Failed to load databases. Try deleting all files in {cache_dir}. Original error: {e}")

orb = utils.feature_engineering(orb)
print(len(orb), "objects loaded")

Loading MPC orbits...
Loading astrometry counts...
Applying nights overrides...
Loading AstDyS orbits...
Loaded 577481 AstDyS orbits
Loading JPL orbits...
Comparing with AstDyS...
Comparing with JPL...
Computing database differences...
Applying number of oppositions overrides...
Applying robust H (median if 3 values, else max (dimmest) across MPC/AstDyS/JPL; corrections applied next)...
Applying magnitude corrections...
Adding training targets...
1559153 objects loaded


In [4]:
extension_difficulty = pd.read_csv("extension_difficulty.csv")
orb = orb.merge(extension_difficulty, on="Principal_desig", how="left")

orb_decent_orbit = orb[((orb["Arc_length"]>=MIN_ARC_LENGTH_FOR_TRAINING)|(orb["Arc_length"].isna())|(orb["Perihelion_dist"]<1.3))
                       &(orb["Num_obs"]>=MIN_NUM_OBS_FOR_TRAINING)
                       &~orb["filtered_out"].astype(bool)
                       &(orb["a_diff_abs"]<0.0005)
                       &(orb["e_diff_abs"]<0.00015)
                       &(orb["i_diff_abs"]<0.003)
                       &(orb["multi_opp_disagree"]==0)
                       &((orb["extension_difficulty"]<0.1)|(orb["extension_difficulty"].isna())|(orb["Perihelion_dist"]<1.3))
                       &((orb["U"]<9)|(orb["Perihelion_dist"]<1.3))
                       &~orb["Principal_desig"].isin(known_strongly_suspected_active_objects)
                       &~orb["Number"].isin(dual_designation_list)]

print(f"Total objects in orbit database: {len(orb)}")
print(f"Total objects in decent orbit database: {len(orb_decent_orbit)}")
orb_training = orb_decent_orbit

Total objects in orbit database: 1559153
Total objects in decent orbit database: 1372602


## Part 2 — Train the three models

Identical to *Start Here* cells 8–13. This is the slow part.

In [5]:
mlcols = ['H', 'Node', 'a', 'i', 'vis_mid',
       'Perihelion_direction_x_e', 'Perihelion_direction_y_e',
       'vis_orbit_mag_multi', 'dec_flux_weighted', 'vis_opp_mean', 'e', 'vis_q', 'vis_timeavg',
       'spatial_discoverability_fraction',
       'Is_Past_Threshold']

mlcols_reg = mlcols.copy()
mlcols_reg.remove("Is_Past_Threshold")
mlcols_reg.append("Num_opps_minus_one")

data_df = orb_training.dropna(subset=["H"])[mlcols].astype(np.float32)
data_df_reg = orb_training.dropna(subset=["H"])[mlcols_reg].astype(np.float32)

shared_folds = np.random.default_rng(0).integers(0, 8, size=len(data_df))
data_df["Shared_Fold"] = shared_folds
data_df_reg["Shared_Fold"] = shared_folds

save_path = os.path.join(tempfile.gettempdir(), 'activityscope_ag')

In [6]:
predictor_reg = TabularPredictor(label="Num_opps_minus_one", groups="Shared_Fold",
    eval_metric=utils.POISSON_SCORER, problem_type='regression',
    path=os.path.join(save_path, datetime.datetime.now().strftime("%Y%m%d_%H%M%S")))
predictor_reg.fit(data_df_reg, presets="good_quality", hyperparameters=utils.HYPERPARAMETERS_POISSON,
    num_stack_levels=0, num_bag_folds=8, dynamic_stacking=False,
    ag_args_ensemble={"fold_fitting_strategy": "sequential_local"}, time_limit=TRAINING_TIME_LIMIT/3)

data_df["exp_Num_opps"] = predictor_reg.predict(data_df) + 1

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jan 19 22:00:55 PST 2026; root:xnu-11417.140.69.708.3~1/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       12.25 GB / 32.00 GB (38.3%)
Disk Space Avail:   44.43 GB / 926.35 GB (4.8%)
Presets specified: ['good_quality']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
Values in c

[1000]	valid_set's poisson: -13.1134	valid_set's mean_poisson_deviance: -0.415474
[1000]	valid_set's poisson: -13.1375	valid_set's mean_poisson_deviance: -0.415597
[1000]	valid_set's poisson: -13.1795	valid_set's mean_poisson_deviance: -0.415032
[1000]	valid_set's poisson: -13.1701	valid_set's mean_poisson_deviance: -0.413391
[1000]	valid_set's poisson: -13.1431	valid_set's mean_poisson_deviance: -0.415927
[1000]	valid_set's poisson: -13.2119	valid_set's mean_poisson_deviance: -0.411855
[1000]	valid_set's poisson: -13.1192	valid_set's mean_poisson_deviance: -0.414936


	-0.4143	 = Validation score   (-mean_poisson_deviance)
	161.34s	 = Training   runtime
	8.15s	 = Validation runtime
Fitting model: XGBoost_BAG_L1 ... Training model for up to 27.69s of the 27.69s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=10, gpus=0)
	-0.4501	 = Validation score   (-mean_poisson_deviance)
	25.31s	 = Training   runtime
	0.55s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 198.44s of the 0.76s of remaining time.
	Fitting 1 model on all data | Fitting with cpus=10, gpus=0, mem=0.1/11.4 GB
	Ensemble Weights: {'LightGBM_BAG_L1': 1.0}
	-0.4143	 = Validation score   (-mean_poisson_deviance)
	0.7s	 = Training   runtime
	0.01s	 = Validation runtime
AutoGluon training complete, total runtime = 200.29s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 21050.9 rows/s (171575 batch size)
Automatically performing refit_full as a post-fit operat

In [7]:
data_df["exp_Num_opps"].update(TabularPredictor.predict_oof(predictor_reg) + 1)
data_df_reg["exp_Num_opps"] = data_df["exp_Num_opps"]

Using OOF from "LightGBM_BAG_L1" as a proxy for "LightGBM_BAG_L1_FULL".
/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_73666/2571831050.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_df["exp_Num_opps"].update(TabularPredictor.predict_oof(predictor_reg) + 1)


In [8]:
predictor_quant = TabularPredictor(label="Num_opps_minus_one", groups="Shared_Fold",
    problem_type='quantile', quantile_levels=[QUANTILE_LEVEL],
    path=os.path.join(save_path, datetime.datetime.now().strftime("%Y%m%d_%H%M%S")))
predictor_quant.fit(data_df_reg, presets="good_quality", hyperparameters=utils.HYPERPARAMETERS_QUANTILE,
    num_stack_levels=1, num_bag_folds=8, dynamic_stacking=False,
    ag_args_ensemble={"fold_fitting_strategy": "sequential_local"}, time_limit=TRAINING_TIME_LIMIT/3)

data_df["quantile_Opps"] = predictor_quant.predict(data_df)[QUANTILE_LEVEL] + 1
data_df["quantile_Opps"].update(TabularPredictor.predict_oof(predictor_quant)[QUANTILE_LEVEL] + 1)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jan 19 22:00:55 PST 2026; root:xnu-11417.140.69.708.3~1/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       11.31 GB / 32.00 GB (35.3%)
Disk Space Avail:   44.22 GB / 926.35 GB (4.8%)
Presets specified: ['good_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
Values in c

In [ ]:
predictor = TabularPredictor(label="Is_Past_Threshold", groups="Shared_Fold", eval_metric='log_loss',
    path=os.path.join(save_path, datetime.datetime.now().strftime("%Y%m%d_%H%M%S")))
predictor.fit(data_df, presets="good_quality", hyperparameters=utils.HYPERPARAMETERS_BINARY,
    num_stack_levels=0, num_bag_folds=8, dynamic_stacking=False,
    ag_args_ensemble={"fold_fitting_strategy": "sequential_local"}, time_limit=TRAINING_TIME_LIMIT/3)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jan 19 22:00:55 PST 2026; root:xnu-11417.140.69.708.3~1/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       11.19 GB / 32.00 GB (35.0%)
Disk Space Avail:   43.96 GB / 926.35 GB (4.7%)
Presets specified: ['good_quality']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
Values in c

## Part 3 — Predictions, `final`, extension difficulty

Identical to *Start Here* cells 15–18.

In [ ]:
# Hard-code both 2009 DP2 and 2024 XE22 to Num_opps = 1 to reproduce the scores
# they had WHEN THE MODEL FLAGGED THEM as single-opp comets.
orb.loc[orb["Principal_desig"]=="2009 DP2",  "Num_opps"] = 1
orb.loc[orb["Principal_desig"]=="2024 XE22", "Num_opps"] = 1
orb.loc[orb["Principal_desig"]=="2009 DP2",  "Arc_length"] = 22
orb.loc[orb["Principal_desig"]=="2024 XE22", "Arc_length"] = 22

In [ ]:
orb_pred = orb.dropna(subset=["H_MPC","H_astdys","H_jpl"], how='all').copy()
orb_pred = utils.apply_nights_overrides(orb_pred)
orb_pred = utils.apply_magnitude_corrections(orb_pred)

orb_pred["exp_Num_opps"] = predictor_reg.predict(orb_pred) + 1
orb_pred["exp_Num_opps"].update(TabularPredictor.predict_oof(predictor_reg) + 1)

orb_pred["quantile_Opps"] = predictor_quant.predict(orb_pred)[QUANTILE_LEVEL] + 1
orb_pred["quantile_Opps"].update(TabularPredictor.predict_oof(predictor_quant)[QUANTILE_LEVEL] + 1)

orb_pred["prob"] = predictor.predict_proba(orb_pred)[1]
orb_pred["prob"].update(TabularPredictor.predict_proba_oof(predictor)[1])

orb_pred["poisson_cdf"] = poisson.cdf(orb_pred["Num_opps"]-1, orb_pred["exp_Num_opps"]-1)

final = utils.getFinal(orb_pred, known_strongly_suspected_active_objects)

In [ ]:
importlib.reload(utils)
orb_pred, final = utils.train_extension_difficulty_classifier(orb_pred, final, orb)

orb_pred = orb_pred[~orb_pred["filtered_out"].astype(bool)]
final = final[~final["filtered_out"].astype(bool)]
print(len(final), "objects in final")

## Part 4 — Candidate pools

`combined_rankings` is the *Start Here* cell 21 pool. Note it is already gated on
$\Delta Q > 0.28\,Q_{0.006}$, which is why Run 4 rebuilds a wider pool without that gate —
otherwise we could never observe $\Delta H$ promoting anything $\Delta Q$ rejects.

In [ ]:
default_col_set = ["prob", "DeltaQ", "poisson_cdf", "extension_difficulty", "Num_opps",
                   "quantile_Opps", "exp_Num_opps", "nights_total", "Ref", "Num_obs",
                   "Arc_length", "U", "a", "i", "H", "q", "TJ"]

final["priority_score"] = final["DeltaQ"] - (final["extension_difficulty"] * 15.5)

combined_rankings = final[((final["Arc_length"]>=8)|(final["Arc_length"].isna()))
      &(final["Num_obs"]>=6)
      &((final["Num_obs"]>=7)|(final["nights_total"]>=5))
      &(final["DeltaQ"]>0.28*final["quantile_Opps"])
      &(final["prob"]>0.8)
      &(final["q"]>1.1)
      &(final["Orbital_period"]<20)].copy()
combined_rankings.sort_values("priority_score", ascending=False, inplace=True)

h_neutrality_candidates = combined_rankings[
    (combined_rankings["prob"] >= 0.998)
    & (combined_rankings["extension_difficulty"] <= 0.15)
    & combined_rankings["Num_opps"].notna()
].head(N_CANDIDATES_RUN1)

print(len(combined_rankings), "in combined_rankings;", len(h_neutrality_candidates), "Delta-H candidates")
combined_rankings.head(15)[['Known / Strong Suspect']+default_col_set]

## Part 5 — Comet file

Needed by Run 2 and Run 5. Identical to *Start Here* cells 38–39. Requires network access
to the MPC.

In [ ]:
discoveries_in_comet_file = ["P/2025 UX109 (Ye)","P/2023 JN16 (Lemmon)","P/2022 BV9 (Lemmon)",
                             "P/2017 FL36 (PANSTARRS)","489P/Denning"]

comet = pd.read_json("https://www.minorplanetcenter.net/Extended_Files/allcometels.json.gz",
                     compression='gzip')
comet = comet[comet["Orbit_type"].isin(["P","D","A"])]
comet["a"] = comet["Perihelion_dist"] / (1 - comet["e"])
comet["Orbital_period"] = np.sqrt(comet["a"]**3)
comet["Tp"] = utils.calc_jd(comet["Year_of_perihelion"], comet["Month_of_perihelion"],
                            comet["Day_of_perihelion"])
comet["Epoch"] = comet["Tp"]
comet["M"] = 0.0

comet_mags_fo = pd.read_fwf("./comet_orbits/comet_els_fo.txt", widths=[8, 5, 1000],
                            names=["Packed", "Mag", "Discard"])

In [ ]:
comet["fullpacked"] = comet["Orbit_type"] + comet["Provisional_packed_desig"]
comet.drop(columns=["H"], inplace=True, errors="ignore")   # comet H, not the inert asteroidal H
cometmerge = comet.merge(comet_mags_fo, left_on="fullpacked", right_on="Packed", how="left")

cometmerge.loc[cometmerge["Designation_and_name"] == "489P/Denning", "Mag"] = 15.35
cometmerge.loc[cometmerge["Designation_and_name"] == "P/2025 UX109 (Ye)", "Mag"] = 15.00

cometmerge["H"] = cometmerge["Mag"]
cometmerge = utils.feature_engineering(cometmerge)
cometmerge.rename(columns={"Perihelion_dist":"q","Aphelion_dist":"Q"}, inplace=True)
print(len(cometmerge), "comets;", cometmerge['H'].notna().sum(), "with usable H")

---
# Run 0 — Is $\lambda(H)$ actually monotone?

**Gates everything else.** Bisection is only valid if the score is non-increasing in $H$.
Gradient-boosted trees give no such guarantee.

**How to read it.** A max violation below ~0.05 opposition is tree-step noise and bisection is
safe. If more than a few percent of objects show real reversals, switch to a coarse grid with
local refinement, or refit with LightGBM/XGBoost `monotone_constraints` on $H$ and the `vis_*`
features (decreasing) and `spatial_discoverability_fraction` (increasing).

The plot also shows the *shape* of $\lambda(H)$ — watch for it flattening at the faint end,
which is what causes $\Delta H$ to saturate and lose resolution among the most extreme objects.

In [ ]:
probe = h_neutrality_candidates.head(30)
grid = np.arange(-2.0, 12.01, 0.5)
C = np.vstack([hn._score_at(probe, np.full(len(probe), d), utils,
                            lambda x: predictor_reg.predict(x) + 1) for d in grid])

viol = np.maximum(np.diff(C, axis=0), 0)
print("max monotonicity violation (oppositions):", round(float(viol.max()), 4))
print("objects with any violation > 0.05 opp:", int((viol.max(axis=0) > 0.05).sum()), "of", C.shape[1])
print("median lambda at delta=0:", round(float(np.median(C[grid==0.0])), 2),
      " at delta=+6:", round(float(np.median(C[grid==6.0])), 2))

plt.figure(figsize=(9,5))
for j in range(C.shape[1]):
    plt.plot(probe["H"].to_numpy()[j] + grid, C[:, j], lw=0.8, alpha=0.7)
plt.axhline(1.0, color='k', ls=':', lw=1, label='$N_{obs}=1$')
plt.xlabel("counterfactual $H_V$"); plt.ylabel("$E[N_{opp}]$")
plt.title("Run 0: expected oppositions vs counterfactual $H$"); plt.legend(); plt.show()

MONOTONE_OK = float(viol.max()) < 0.05
print("\nMONOTONE_OK =", MONOTONE_OK)

---
# Run 1 — The $\Delta H$ table (replaces cell 22)

Answers **Q2**: what the principled definitions actually produce.

**How to read it.**
- `dH_Q` vs `dH_E`: the bracket. $\Delta H_Q$ = "leaves the 0.006 tail", $\Delta H_E$ = "fully
  reconciled". A wide gap means the choice of operating point matters; a narrow one means it doesn't.
- `dH_*_censored`: fraction that never reconcile within `DELTA_MAX`. **If this is common, $\Delta H$
  is saturating and losing resolution at the top of the list** — that is the main way this idea
  could fail as a ranker.
- `Z_H`: significance against photometric scatter. $Z_H > 3$ is the referee's
  "needs more explanation than photometric scatter".
- `exp_Num_opps_full` vs `exp_Num_opps`: the refit-vs-OOF offset. If large, it needs a sentence
  in the paper, because $\Delta H$ is measured against the refit model while Table 3's anomaly
  is out-of-fold.

> **Cascade.** Your pipeline feeds regression output into the quantile model as the
> `exp_Num_opps` feature, and feeds both `exp_Num_opps` and `quantile_Opps` into the binary
> classifier. Those columns are therefore **H-dependent inputs**. A sweep that leaves them at
> their catalog values while changing $H$ silently pins part of the model, biasing
> $\Delta H_Q$ and $\Delta H_P$ low. `h_neutrality.py` recomputes the upstream predictions at
> every trial $H$. The existing cell 22 is unaffected because `predictor_reg` has no such
> dependency — but any hand-rolled sweep through the quantile or binary model would be.

In [ ]:
dh = hn.h_neutrality_table(
    h_neutrality_candidates, utils, predictor_reg, predictor_quant, predictor,
    quantile_level=QUANTILE_LEVEL, p_star=P_STAR, sigma_h=SIGMA_H,
    delta_max=DELTA_MAX, n_iter=N_BISECT,
    extra_columns=("extension_difficulty","prob","DeltaQ","poisson_cdf",
                   "priority_score","Known / Strong Suspect","a","TJ","q"))

dh["exp_Num_opps_oof"] = h_neutrality_candidates["exp_Num_opps"]
dh["oof_refit_offset"]  = dh["exp_Num_opps_full"] - dh["exp_Num_opps_oof"]

print("censored dH_Q: %.1f%%   censored dH_E: %.1f%%   invalid: %d"
      % (100*dh["dH_Q_censored"].mean(), 100*dh["dH_E_censored"].mean(), int(dh["dH_E_invalid"].sum())))
print("median dH_Q = %.2f mag, median dH_E = %.2f mag, median gap = %.2f mag"
      % (dh["dH_Q"].median(), dh["dH_E"].median(), (dh["dH_E"]-dh["dH_Q"]).median()))
print("median |refit - OOF| offset in E[N_opp]: %.2f opp" % dh["oof_refit_offset"].abs().median())
print("fraction with Z_H > 3: %.2f" % (dh["Z_H"] > 3).mean())

dh.to_csv(f"{RESULTS_DIR}/run1_delta_h_table.csv")
dh[["H_current","H_neutral","dH_Q","dH_E","Z_H","Num_opps","DeltaQ","prob",
    "extension_difficulty","Known / Strong Suspect"]].round(2).head(30)

---
# Run 2 — DECISIVE: comet holdout

Answers **Q1** without selection contamination.

The known-active list is contaminated: most of those objects were *found* by $\Delta Q$ ranking,
so $\Delta Q$ has home-field advantage there. The comet file does not have that problem — these
are genuinely active objects that no ActivitySCOPE score ever selected. We score them with the
asteroidal $H_V$ they would have had, set $N_{\rm opp}=1$ (the paper's own counterfactual: "these
would have been flagged as soon as the single-opposition linkage was complete"), and rank them
against real single-opposition objects on comparable orbits.

**Two caveats you must carry into the interpretation.**
1. Absolute AUC is **inflated** — comets and single-opp asteroids don't occupy identical orbital
   space even after the $T_J$ and $a$ cuts, so some signal is just "is this a cometary orbit."
   The confound hits all six metrics equally, so read the *differences between metrics*, not the
   absolute values.
2. Comet $H$ comes from find_orb (`comet_els_fo.txt`), a different photometric footing than
   MPCORB. Worth a sentence if this benchmark goes in the paper.

**Decision rule.** If `dH_Q` matches or beats `DeltaQ_refit` on both `auc` and
`interlopers_above`, the Q3 simplification is safe. If it loses badly at the head of the list,
keep $\Delta Q$ as the ranker and use $\Delta H$ only as the Table 3 interpretability column.

In [ ]:
cm = cometmerge.dropna(subset=["H"]).copy()
cm = cm[~cm["Designation_and_name"].duplicated()].set_index("Designation_and_name")
cm["Num_opps"] = 1.0
cm["is_active"] = True
pos = cm[(cm["TJ"] < 3.05) & (cm["q"] > 1.1) & cm["a"].between(1.1, 9)]

neg = final[(final["Num_opps"] == 1) & (final["TJ"] < 3.05) & (final["q"] > 1.1)
            & final["a"].between(1.1, 9) & (final["extension_difficulty"] < 0.25)
            & ~final["Known / Strong Suspect"] & final["H"].notna()].copy()
if len(neg) > N_NEGATIVES_RUN2:
    neg = neg.sample(N_NEGATIVES_RUN2, random_state=RNG_SEED)
neg["is_active"] = False

print(f"{len(pos)} comet positives, {len(neg)} single-opp asteroid negatives")
pool2 = pd.concat([pos, neg], axis=0)

In [ ]:
res2 = hn.h_neutrality_table(pool2, utils, predictor_reg, predictor_quant, predictor,
                             quantile_level=QUANTILE_LEVEL, p_star=P_STAR, sigma_h=SIGMA_H,
                             delta_max=DELTA_MAX, n_iter=N_BISECT)
res2["is_active"]     = pool2["is_active"]
res2["DeltaQ_refit"]  = res2["quantile_Opps_full"] - res2["Num_opps"]
res2["S_EV_refit"]    = poisson.cdf(res2["Num_opps"] - 1, res2["exp_Num_opps_full"] - 1)

METRICS = {"dH_Q": ("dH_Q", True), "dH_E": ("dH_E", True), "Z_H": ("Z_H", True),
           "DeltaQ": ("DeltaQ_refit", True), "S_EV": ("S_EV_refit", False),
           "P(N>=4)": ("prob_full", True)}

cmp2 = hev.compare_rankings(res2, "is_active", METRICS)
res2.to_csv(f"{RESULTS_DIR}/run2_comet_holdout.csv")
cmp2.to_csv(f"{RESULTS_DIR}/run2_comparison.csv")
cmp2.round(4)

In [ ]:
# Verdict for Q1
if {"dH_Q","DeltaQ"} <= set(cmp2.index):
    a_new, a_old = cmp2.loc["dH_Q","auc"], cmp2.loc["DeltaQ","auc"]
    i_new, i_old = cmp2.loc["dH_Q","interlopers_above"], cmp2.loc["DeltaQ","interlopers_above"]
    print(f"AUC:          dH_Q = {a_new:.4f}   DeltaQ = {a_old:.4f}   delta = {a_new-a_old:+.4f}")
    print(f"Interlopers:  dH_Q = {i_new:.0f}       DeltaQ = {i_old:.0f}")
    Q1_VERDICT = ("dH matches or beats DeltaQ" if (a_new >= a_old - 0.005 and i_new <= i_old)
                  else "dH is worse - keep DeltaQ as the ranker")
    print("\nQ1 VERDICT:", Q1_VERDICT)

---
# Run 3 — Known-active separability (supporting, biased)

This reproduces the paper's own "16 of the top 18" claim as a number, for each metric.

**Read this as a floor on $\Delta H$'s performance, not a fair fight** — the positives here were
largely discovered by $\Delta Q$ ranking. `interlopers_above` is the direct analogue of the
paper's claim.

`dH_priority` is the priority score rebuilt in magnitudes. Unlike the current $15.5$, the
coefficient now means something: "magnitudes of counterfactual brightening I will trade for one
unit of linkage risk." The loop tunes it.

In [ ]:
cr = combined_rankings[combined_rankings["Num_opps"].notna()].head(400).copy()
res3 = hn.h_neutrality_table(cr, utils, predictor_reg, predictor_quant, predictor,
                             quantile_level=QUANTILE_LEVEL, p_star=P_STAR, sigma_h=SIGMA_H,
                             delta_max=DELTA_MAX, n_iter=N_BISECT)
res3["active"]         = cr["Known / Strong Suspect"]
res3["DeltaQ"]         = cr["DeltaQ"]
res3["priority_score"] = cr["priority_score"]
res3["ext_diff"]       = cr["extension_difficulty"]

best_k, best_auc = None, -1
for k in np.arange(0.0, 4.01, 0.25):
    auc = hev.roc_auc(res3["dH_Q"] - k*res3["ext_diff"], res3["active"], True)
    if auc > best_auc: best_k, best_auc = k, auc
res3["dH_priority"] = res3["dH_Q"] - best_k*res3["ext_diff"]
print(f"best k = {best_k:.2f} mag per unit extension_difficulty  (auc {best_auc:.4f})")

cmp3 = hev.compare_rankings(res3, "active", {
    "dH_Q": ("dH_Q", True), "dH_E": ("dH_E", True), "dH_priority": ("dH_priority", True),
    "DeltaQ": ("DeltaQ", True), "priority_score": ("priority_score", True)})
res3.to_csv(f"{RESULTS_DIR}/run3_known_active.csv")
cmp3.round(4)

---
# Run 4 — Objects $\Delta Q$ structurally cannot reach

Answers **Q4**.

$\Delta Q \ge 3$ is unreachable when the predicted count is small — exactly the failure the paper
already documents for 2009 DP$_2$ ("a two-opposition object with averaged $H_V=15.1$ would not be
strongly flagged, suggesting that some sub-investigation-threshold objects may be missed").
$\Delta H$ has no such floor because it is scale-free in $N_{\rm opp}$.

This rebuilds the pool **without** the $\Delta Q$ gate and asks what $\Delta H$ promotes that
$\Delta Q$ rejects. If this idea finds anything new, it shows up here — and the prediction is
DP$_2$-like objects: multi-opposition, distant, small predicted counts.

In [ ]:
wide = final[((final["Arc_length"]>=8)|(final["Arc_length"].isna()))
      & (final["Num_obs"]>=6) & ((final["Num_obs"]>=7)|(final["nights_total"]>=5))
      & (final["prob"]>0.8) & (final["q"]>1.1)
      & (final["extension_difficulty"]<0.15) & (final["Orbital_period"]<20)
      & final["Num_opps"].notna() & final["H"].notna()].copy()
print(len(wide), "in wide pool before capping")
if len(wide) > N_WIDE_RUN4:
    wide = wide.nlargest(N_WIDE_RUN4, "prob")

res4 = hn.h_neutrality_table(wide, utils, predictor_reg, predictor_quant, predictor,
                             quantile_level=QUANTILE_LEVEL, p_star=P_STAR, sigma_h=SIGMA_H,
                             delta_max=DELTA_MAX, n_iter=N_BISECT,
                             extra_columns=("DeltaQ","prob","extension_difficulty","a","TJ","q","H",
                                            "Known / Strong Suspect","Num_obs","Arc_length"))
res4.to_csv(f"{RESULTS_DIR}/run4_wide_pool.csv")

new_finds = hev.marginal_discoveries(res4, "dH_Q", "DeltaQ", old_threshold=3, top_n=100)
print(f"\n{len(new_finds)} of the top-100 dH_Q objects have DeltaQ < 3 "
      f"({new_finds['Known / Strong Suspect'].sum()} already known active)")
new_finds[["H_current","H_neutral","dH_Q","dH_E","Z_H","Num_opps","DeltaQ","prob",
           "extension_difficulty","a","TJ","Known / Strong Suspect"]].round(2)

---
# Run 5 — The archival ladder

Answers the referee directly, and is the figure I would build the response around.

The paper already states archival non-detections in $\Delta H$ units — "$N$ mag fainter than
predicted" converts to $H_{\rm limit} = H_{\rm current} + N$, since the prediction uses the
catalog $H$. Where $H_{\rm limit} \ge H_{\rm neutral}$, **the brightening the model demands was
independently observed at another epoch**, so the deficit needs no photometric-scatter
explanation at all.

⚠️ **Verify every row of the dict below against your own notes.** I read these off `main.tex`;
only 2008 BJ$_{22}$ and 2021 AY$_8$ are stated there as explicit $H_V$ values, the rest are
derived from "$N$ mag fainter than predicted" phrasing and should be checked.

In [ ]:
# Explicit H_V limits stated in the paper
ARCHIVAL_H_EXPLICIT = {
    "2008 BJ22": 19.3,      # H_V > 18.4 (2005), > 19.3 (2013)
    "2021 AY8":  19.9,      # quiescent H_V ~ 19.9 vs 17.5 in outburst
}
# Derived from "N mag fainter than predicted" -> H_limit = H_current + N
ARCHIVAL_DELTA_MAG = {
    "2010 RH69":   3.0,     # >=2.5 (2008), ~3 (2016)
    "2024 XE22":   3.0,     # 2 (2022), 3 (2009)
    "2002 CW116":  2.0,     # 2.0 (2005), 2.0 (2010)
    "2010 TR241":  2.5,     # ~0.5 (2011), ~2.5 (2013 deep)
    "2009 FP8":    1.2,     # predicted 20.6, limit 21.8 (2014)
    "2015 BC566":  1.0,     # 1.0 (2014 Dec 15)
    "2018 BJ11":   1.0,     # ~1 (2019)
    "2001 BV70":   1.0,     # ~1 (2005)
}

TABLE3_OBJECTS = sorted(set(ARCHIVAL_H_EXPLICIT) | set(ARCHIVAL_DELTA_MAG) |
                        {"2025 HV38","2007 VB146","2008 GO98","2019 OE31","2025 VZ8",
                         "2009 DP2","2008 VK110","2017 QN84","2003 BM80","2007 HE4"})

present = [d for d in TABLE3_OBJECTS if d in final.index]
missing = [d for d in TABLE3_OBJECTS if d not in final.index]
print("in final:", len(present), " | not in final (likely in the comet file):", missing)

t3 = final.loc[present]
t3 = t3[t3["H"].notna() & t3["Num_opps"].notna()]
dh3 = hn.h_neutrality_table(t3, utils, predictor_reg, predictor_quant, predictor,
                            quantile_level=QUANTILE_LEVEL, p_star=P_STAR, sigma_h=SIGMA_H,
                            delta_max=DELTA_MAX, n_iter=N_BISECT,
                            extra_columns=("DeltaQ","prob","poisson_cdf","a","TJ","q"))

limits = dict(ARCHIVAL_H_EXPLICIT)
for d, dmag in ARCHIVAL_DELTA_MAG.items():
    if d in dh3.index:
        limits[d] = float(dh3.loc[d, "H_current"]) + dmag

ladder = hn.archival_consistency(dh3, limits)
ladder.to_csv(f"{RESULTS_DIR}/run5_archival_ladder.csv")
ladder[["H_current","H_neutral","dH_Q","dH_E","Z_H","H_archival_limit",
        "archival_margin","archival_supports_dH"]].round(2).sort_values("dH_Q", ascending=False)

In [ ]:
# The figure: catalog H, required H_neutral, and the measured archival limit
lad = ladder.dropna(subset=["H_archival_limit"]).sort_values("dH_Q")
fig, ax = plt.subplots(figsize=(9, 0.45*len(lad)+2))
y = np.arange(len(lad))
ax.hlines(y, lad["H_current"], lad["H_neutral"], color="0.6", lw=3,
          label=r"required brightening $\Delta H_Q$")
ax.plot(lad["H_current"], y, "o", color="tab:red", label="catalog $H_V$ (as discovered)")
ax.plot(lad["H_neutral"], y, "s", color="tab:blue", label="$H_V$ at which anomaly vanishes")
ax.plot(lad["H_archival_limit"], y, "v", color="tab:green",
        label="archival non-detection limit")
ax.set_yticks(y); ax.set_yticklabels(lad.index)
ax.set_xlabel("$H_V$ (mag)"); ax.invert_xaxis()
ax.set_title("Run 5: required brightening vs. independently measured quiescent limits")
ax.legend(loc="best", fontsize=9); plt.tight_layout(); plt.show()

n_ok = int(ladder["archival_supports_dH"].sum())
print(f"{n_ok} of {int(ladder['H_archival_limit'].notna().sum())} objects with archival limits "
      f"have the required brightening independently corroborated (green marker at or beyond blue).")

---
# Run 6 — Can $\Delta H_E$ replace both $\Delta Q$ *and* $S_{\rm EV}$?

Answers **Q3**, the structural question behind the simplification.

$S_{\rm EV}$ exists because multi-opposition ACOs and Centaurs — 2008 VK$_{110}$ is the paper's
example — surface only against $E[N_{\rm opp}]$, not against the quantile, since the quantile
regressor is data-starved out there. $\Delta H_E$ is built on $E[N_{\rm opp}]$, so it *should*
inherit that property. If it does, the quantile model can be retired entirely.

**Two tests.**
1. Rank correlation among the metrics. High $\Delta H_Q$–$\Delta Q$ correlation is *expected*
   (both are monotone functionals of the same surface at the same point) and is not a problem —
   it means $\Delta H$ is a reparameterization, which is the claim.
2. Where the $S_{\rm EV}$-only discoveries land under $\Delta H_E$ among multi-opposition
   objects. If they rank highly, $\Delta H_E$ subsumes $S_{\rm EV}$.

In [ ]:
sub = res4.dropna(subset=["dH_Q","dH_E","DeltaQ"])
pairs = [("dH_Q","dH_E"), ("dH_Q","DeltaQ"), ("dH_E","DeltaQ")]
for x, y in pairs:
    rho = spearmanr(sub[x], sub[y]).statistic
    print(f"Spearman rho({x:>6}, {y:>6}) = {rho:+.3f}")

SEV_ONLY = ["2008 VK110","2009 DP2","2010 RH69","2019 OE31","2018 BJ11","2003 BM80"]
multi = final[(final["Num_opps"] >= 2) & (final["TJ"] < 3.05) & final["H"].notna()
              & (final["extension_difficulty"] < 0.6) & (final["q"] > 1.1)
              & final["a"].between(1.1, 9)].copy()
keep = [d for d in SEV_ONLY if d in multi.index]
others = multi.drop(index=keep).nlargest(min(600, len(multi)), "prob")
pool6 = pd.concat([multi.loc[keep], others])
print(f"\nmulti-opp pool: {len(pool6)}; S_EV-only objects present: {keep}")

res6 = hn.h_neutrality_table(pool6, utils, predictor_reg, predictor_quant,
                             quantile_level=QUANTILE_LEVEL, sigma_h=SIGMA_H,
                             delta_max=DELTA_MAX, n_iter=N_BISECT,
                             extra_columns=("DeltaQ","poisson_cdf","prob","a","TJ"))
res6["S_EV_refit"] = poisson.cdf(res6["Num_opps"]-1, res6["exp_Num_opps_full"]-1)

for metric, asc in [("dH_E", False), ("dH_Q", False), ("DeltaQ", False), ("S_EV_refit", True)]:
    ranked = res6.sort_values(metric, ascending=asc)
    pos = {d: int(ranked.index.get_loc(d))+1 for d in keep if d in ranked.index}
    print(f"{metric:>11}: ranks {pos}")

res6.to_csv(f"{RESULTS_DIR}/run6_multiopp.csv")

---
# Synthesis — the four answers

Run this last. It collects the numbers above into a direct answer to each question.

In [ ]:
print("="*78)
print("Q1. Does dH separate active from background as well as DeltaQ?")
print("="*78)
print("  Comet holdout (uncontaminated, DECISIVE):")
print(cmp2[["auc","interlopers_above","precision_at_n_pos"]].round(4).to_string())
print("\n  Known-active list (contaminated, favours DeltaQ - read as a floor):")
print(cmp3[["auc","interlopers_above","precision_at_n_pos"]].round(4).to_string())

print("\n" + "="*78)
print("Q2. Is there a more principled definition than the 0.5 tolerance?")
print("="*78)
print(f"  Yes: three parameter-free crossing points. Monotonicity holds: {MONOTONE_OK}")
print(f"  (max violation {float(viol.max()):.4f} opp)")
print(f"  median dH_Q = {dh['dH_Q'].median():.2f} mag, median dH_E = {dh['dH_E'].median():.2f} mag")
print(f"  censored: dH_Q {100*dh['dH_Q_censored'].mean():.1f}%, dH_E {100*dh['dH_E_censored'].mean():.1f}%")
print(f"  -> if censoring is high, dH saturates at the top of the list and loses resolution")
print(f"  fraction of candidates with Z_H > 3 (beyond photometric scatter): {(dh['Z_H']>3).mean():.2f}")

print("\n" + "="*78)
print("Q3. How much simpler would the paper be?")
print("="*78)
print("  Structural question: does dH_E subsume S_EV on multi-opp ACOs? See Run 6 ranks.")
print("  If yes, the quantile model retires and you lose:")
print("    - the Quantile Regression Model paragraph in 2.5, and its OOF plumbing")
print("    - one of three paragraphs in 2.6, incl. the 'S_EV is not a probability' caveat")
print("    - 2 columns + 2 footnotes from Table 3 (net -1 after adding dH)")
print("    - one of three trained models")
print("  Cost: you lose 'distribution-agnostic' as a selling point, and dH is ~14x the cost")
print("  of a single prediction, so it is a second-stage re-ranker, not a catalog-wide filter.")

print("\n" + "="*78)
print("Q4. Can it find objects we previously missed?")
print("="*78)
print(f"  {len(new_finds)} of the top-100 dH_Q objects have DeltaQ < 3 "
      f"({int(new_finds['Known / Strong Suspect'].sum())} already known active,"
      f" {len(new_finds) - int(new_finds['Known / Strong Suspect'].sum())} not)")
print(f"  Archival ladder: {n_ok} objects have their required brightening independently")
print(f"  corroborated by measured non-detection limits.")
print("\nAll CSVs written to ./" + RESULTS_DIR + "/")